# EXP-00: MNIST Baseline 실습

기본 CNN 모델로 MNIST를 학습하고, PC 폰트로 실전 테스트까지 진행합니다.

**실행 환경**: Google Colab (GPU 권장, CPU도 가능)

## 1. 환경 설정

In [ ]:
# Google Drive 마운트
from google.colab import drive
drive.mount('/content/drive')

# 프로젝트 폴더 생성
import os
PROJECT_PATH = '/content/drive/MyDrive/ai-practice/01-MNIST'
os.makedirs(f'{PROJECT_PATH}/models', exist_ok=True)
os.makedirs(f'{PROJECT_PATH}/results', exist_ok=True)
print(f"프로젝트 경로: {PROJECT_PATH}")

In [ ]:
# 환경 확인
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np

print(f"PyTorch 버전: {torch.__version__}")
print(f"CUDA 사용 가능: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    device = torch.device('cuda')
else:
    print("CPU 사용")
    device = torch.device('cpu')

# 재현성
torch.manual_seed(42)

## 2. 데이터 로드

In [ ]:
# 전처리 정의
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

# 데이터셋 로드
train_dataset = datasets.MNIST('./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST('./data', train=False, download=True, transform=transform)

# DataLoader
BATCH_SIZE = 64
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=1000, shuffle=False)

print(f"학습 데이터: {len(train_dataset):,}장")
print(f"테스트 데이터: {len(test_dataset):,}장")

In [ ]:
# 샘플 시각화
fig, axes = plt.subplots(2, 5, figsize=(12, 5))

for i, ax in enumerate(axes.flat):
    image, label = train_dataset[i]
    img = image.squeeze().numpy()
    img = img * 0.3081 + 0.1307
    ax.imshow(img, cmap='gray')
    ax.set_title(f'Label: {label}')
    ax.axis('off')

plt.suptitle('MNIST 샘플 이미지')
plt.tight_layout()
plt.show()

## 3. 모델 정의

In [ ]:
class CNN(nn.Module):
    """
    간단한 CNN 모델
    Conv1(32) → Pool → Conv2(64) → Pool → FC(128) → FC(10)
    """
    def __init__(self):
        super(CNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 10)
        self.dropout = nn.Dropout(0.25)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(-1, 64 * 7 * 7)
        x = self.dropout(F.relu(self.fc1(x)))
        x = self.fc2(x)
        return x

model = CNN().to(device)
print(model)
print(f"\n총 파라미터 수: {sum(p.numel() for p in model.parameters()):,}")

## 4. 학습

In [ ]:
# 하이퍼파라미터
EPOCHS = 10
LEARNING_RATE = 0.001

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

print(f"에포크: {EPOCHS}")
print(f"학습률: {LEARNING_RATE}")

In [ ]:
def train_epoch(model, device, train_loader, optimizer, criterion):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for data, target in train_loader:
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        pred = output.argmax(dim=1)
        correct += pred.eq(target).sum().item()
        total += target.size(0)
    
    return total_loss / len(train_loader), 100. * correct / total

def evaluate(model, device, test_loader, criterion):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            total_loss += criterion(output, target).item()
            pred = output.argmax(dim=1)
            correct += pred.eq(target).sum().item()
            total += target.size(0)
    
    return total_loss / len(test_loader), 100. * correct / total

In [ ]:
# 학습 실행
history = {'train_loss': [], 'train_acc': [], 'test_loss': [], 'test_acc': []}

print("=" * 50)
print("학습 시작")
print("=" * 50)

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = train_epoch(model, device, train_loader, optimizer, criterion)
    test_loss, test_acc = evaluate(model, device, test_loader, criterion)
    
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['test_loss'].append(test_loss)
    history['test_acc'].append(test_acc)
    
    print(f"Epoch {epoch:2d}/{EPOCHS} | "
          f"Train Loss: {train_loss:.4f}, Acc: {train_acc:.2f}% | "
          f"Test Loss: {test_loss:.4f}, Acc: {test_acc:.2f}%")

print("\n" + "=" * 50)
print(f"학습 완료! 최종 테스트 정확도: {history['test_acc'][-1]:.2f}%")
print("=" * 50)

## 5. 학습 곡선 시각화

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history['train_loss'], 'b-', label='Train', linewidth=2)
axes[0].plot(history['test_loss'], 'r-', label='Test', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Loss Curve')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history['train_acc'], 'b-', label='Train', linewidth=2)
axes[1].plot(history['test_acc'], 'r-', label='Test', linewidth=2)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('Accuracy Curve')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{PROJECT_PATH}/results/learning_curves.png', dpi=150)
plt.show()

## 6. 혼동 행렬

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns

model.eval()
all_preds, all_targets = [], []

with torch.no_grad():
    for data, target in test_loader:
        data = data.to(device)
        output = model(data)
        pred = output.argmax(dim=1).cpu().numpy()
        all_preds.extend(pred)
        all_targets.extend(target.numpy())

cm = confusion_matrix(all_targets, all_preds)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=range(10), yticklabels=range(10))
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.savefig(f'{PROJECT_PATH}/results/confusion_matrix.png', dpi=150)
plt.show()

print("\n클래스별 정확도:")
for i in range(10):
    class_acc = cm[i, i] / cm[i].sum() * 100
    print(f"  숫자 {i}: {class_acc:.1f}%")

## 7. 모델 저장

In [ ]:
# 모델 저장
model_path = f'{PROJECT_PATH}/models/baseline_cnn.pt'
torch.save({
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'history': history,
    'final_accuracy': history['test_acc'][-1]
}, model_path)

print(f"모델 저장 완료: {model_path}")

# 가중치만 저장
weights_path = f'{PROJECT_PATH}/models/baseline_weights.pth'
torch.save(model.state_dict(), weights_path)
print(f"가중치 저장 완료: {weights_path}")

## 8. PC 폰트 테스트 (로컬 전용)

이 섹션은 **로컬 환경**에서 실행해야 합니다.
1. 위에서 저장한 모델을 로컬로 다운로드
2. 아래 코드를 로컬 Python 환경에서 실행

In [ ]:
# ============================================================
# 이 셀은 로컬 환경에서 실행하세요!
# Colab에서는 Windows 폰트에 접근할 수 없습니다.
# ============================================================

# 로컬 실행 여부 확인
import sys
if 'google.colab' in sys.modules:
    print("⚠️ 이 셀은 로컬 환경에서 실행해야 합니다!")
    print("모델을 다운로드하고 로컬에서 font_test.py를 실행하세요.")
else:
    print("✓ 로컬 환경 확인됨. 폰트 테스트를 진행합니다.")

## 9. 결과 요약

In [ ]:
import json
from datetime import datetime

result = {
    'experiment': 'EXP-00 Baseline',
    'date': datetime.now().strftime('%Y-%m-%d %H:%M'),
    'model': 'CNN (32-64-128-10)',
    'epochs': EPOCHS,
    'learning_rate': LEARNING_RATE,
    'batch_size': BATCH_SIZE,
    'final_train_acc': history['train_acc'][-1],
    'final_test_acc': history['test_acc'][-1],
    'final_train_loss': history['train_loss'][-1],
    'final_test_loss': history['test_loss'][-1]
}

with open(f'{PROJECT_PATH}/results/exp00_results.json', 'w') as f:
    json.dump(result, f, indent=2)

print("\n" + "=" * 50)
print("실험 결과 요약")
print("=" * 50)
for key, value in result.items():
    if isinstance(value, float):
        print(f"{key}: {value:.4f}")
    else:
        print(f"{key}: {value}")

---

## 다음 단계

1. 로컬에서 `font_test.py`로 PC 폰트 테스트
2. EXP-01: 모델 구조 비교 (MLP vs CNN)
3. EXP-02: 활성화 함수 비교